In [2]:
######################################################
#environment
######################################################

In [3]:
import os

# directories
os.environ["BASE_DIR"] = "/kaggle/working/megarag_vdoc"
os.environ["TMP_DIR"] = "/kaggle/tmp/megarag_vdoc"
os.environ["OUT_DIR"] = "/kaggle/working/megarag_outputs"

# Hugging Face cache -> /kaggle/tmp 
os.environ["HF_HOME"] = "/kaggle/tmp/megarag_vdoc/hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/tmp/megarag_vdoc/hf/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/megarag_vdoc/hf/models"

# resource
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Embedding GME-Qwen2-VL-2B -> chạy bằng CPU
# chạy GPU -> đổi thành "cuda"
os.environ["EMBED_DEVICE"] = "cpu"

# OCR bằng Qwen3-VL trước khi build MMKG
# chạy nhanh hơn -> đổi thành "0"
os.environ["USE_OCR"] = "1"

# ====== Sample subset ======
os.environ["SAMPLE_PAGES"] = "30" # "40"
os.environ["MAX_QUESTIONS_PER_PAGE"] = "3" #"4"
os.environ["MAX_EVAL_QUESTIONS"] = "90" # "120"

#  dùng LLM-judge sau khi query
os.environ["RUN_LLM_JUDGE"] = "1"
os.environ["JUDGE_SAMPLE"] = "120"

# Baseline no-RAG -> dùng so sánh
os.environ["RUN_BASELINE"] = "1"

# Config Qwen
os.environ["QWEN_MAX_PIXELS"] = str(768 * 28 * 28)   # giảm độ phân giải
os.environ["QWEN_MAX_INPUT_CHARS"] = "24000"

# Tạo thư mục
for d in [
    os.environ["BASE_DIR"],
    os.environ["TMP_DIR"],
    os.environ["OUT_DIR"],
    os.environ["HF_HOME"],
    os.environ["HF_DATASETS_CACHE"],
    os.environ["TRANSFORMERS_CACHE"],
]:
    os.makedirs(d, exist_ok=True)

print("Done env")

Done env


In [4]:
######################################################
# Clone MegaRAG + Install dependencies
######################################################

In [5]:
%%bash
set -e

mkdir -p /kaggle/working/megarag_vdoc
mkdir -p /kaggle/tmp/megarag_vdoc

cd /kaggle/working/megarag_vdoc

# Clone MegaRAG
if [ ! -d "MegaRAG" ]; then
  git clone https://github.com/AI-Application-and-Integration-Lab/MegaRAG.git
fi

cd MegaRAG

python -m pip install -U pip wheel

# Qwen3-VL -> require transformers >= 4.57 
python -m pip install --no-cache-dir \
  "transformers>=4.57.0,<5.0" \
  accelerate \
  bitsandbytes \
  sentencepiece \
  protobuf \
  datasets \
  pillow \
  pyyaml \
  tqdm \
  tenacity \
  openai \
  nano-vectordb \
  pipmaster \
  python-dotenv \
  pydantic \
  qwen_vl_utils

# LightRAG
python -m pip install --no-cache-dir git+https://github.com/HKUDS/LightRAG.git@v1.4.3

# Cài MegaRAG (editable)
python -m pip install --no-cache-dir -e . --no-deps

python -m pip install --no-cache-dir uvloop || true

echo "Installed done."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.4 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 130.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 198.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 211.3 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
   

Cloning into 'MegaRAG'...
  Running command git clone --filter=blob:none --quiet https://github.com/HKUDS/LightRAG.git /tmp/pip-req-build-v0mdsch8
  Running command git checkout -q 0171e0ce20e7b4415929d01b634732067d206a62


In [6]:
######################################################
# Create qwen_llm.py -> Qwen3 VL
######################################################

In [7]:
%%writefile /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/qwen_llm.py
import os
import asyncio
import threading
import logging
from typing import List, Optional, Any
import torch

MODEL_ID = os.environ.get(
    "QWEN_MODEL_ID",
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit"
)

_model = None
_processor = None
_load_lock = threading.Lock()
_gen_lock = threading.Lock()
logger = logging.getLogger("qwen_llm")

MAX_INPUT_CHARS = int(os.environ.get("QWEN_MAX_INPUT_CHARS", "24000"))


def _free_gpu():
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def _load_model():
    global _model, _processor

    with _load_lock:
        if _model is not None:
            return _model, _processor

        from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is required for Qwen3-VL in this notebook.")

        n_gpu = torch.cuda.device_count()
        print(f"[qwen_llm] Detected {n_gpu} GPU(s): "
              f"{[torch.cuda.get_device_name(i) for i in range(n_gpu)]}")

        try:
            # device_map="auto" -> trải model trên tất cả GPU (2x T4 = ~32GB)
            _model = Qwen3VLForConditionalGeneration.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.float16,
                attn_implementation="sdpa",
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True,
            )
        except Exception as e:
            logger.warning(f"Loading with sdpa failed: {e}; fallback to default attention.")
            _model = Qwen3VLForConditionalGeneration.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True,
            )

        max_pixels = int(os.environ.get("QWEN_MAX_PIXELS", str(768 * 28 * 28)))
        try:
            _processor = AutoProcessor.from_pretrained(
                MODEL_ID,
                min_pixels=256 * 28 * 28,
                max_pixels=max_pixels,
                trust_remote_code=True,
            )
        except TypeError:
            _processor = AutoProcessor.from_pretrained(
                MODEL_ID,
                trust_remote_code=True
            )

        _model.eval()
        return _model, _processor


def _normalize_content(content):
    if content is None:
        return [{"type": "text", "text": ""}]
    if isinstance(content, str):
        return [{"type": "text", "text": content}]
    if isinstance(content, list):
        return content
    return [{"type": "text", "text": str(content)}]


def _truncate_text(text, max_chars):
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...(bị cắt bớt do quá dài)"


def _build_messages(prompt, input_images=None, system_prompt=None, history_messages=None):
    messages = []

    if system_prompt:
        messages.append(
            {
                "role": "system",
                "content": [{"type": "text", "text": str(system_prompt)[:2000]}],
            }
        )

    if history_messages:
        for msg in history_messages[-4:]:
            if not isinstance(msg, dict):
                continue
            role = msg.get("role", "user")
            content = msg.get("content", "")
            messages.append(
                {
                    "role": role,
                    "content": _normalize_content(content),
                }
            )

    user_content = []

    if input_images:
        # tối đa 3 ảnh / lần gọi -> tránh quá tải vision tokens
        for img in input_images[:3]:
            if not img:
                continue
            if isinstance(img, str):
                if img.startswith("http://") or img.startswith("https://") or os.path.exists(img):
                    user_content.append({"type": "image", "image": img})

    user_content.append({"type": "text", "text": _truncate_text(prompt, MAX_INPUT_CHARS)})
    messages.append({"role": "user", "content": user_content})

    return messages


def _generate_once(messages, max_new_tokens):
    model, processor = _load_model()

    with torch.inference_mode():
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )

        inputs = {
            k: (v.to(model.device) if hasattr(v, "to") else v)
            for k, v in inputs.items()
        }

        input_len = int(inputs["input_ids"].shape[1])

        pad_token_id = processor.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = processor.tokenizer.eos_token_id

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_token_id,
            use_cache=True,
        )

        output_ids = generated_ids[0, input_len:]
        text = processor.batch_decode([output_ids], skip_special_tokens=True)[0]

    # Giải phóng input tensors
    del inputs, generated_ids
    _free_gpu()

    return text.strip(), input_len, int(output_ids.shape[0])


def _generate_sync(
    prompt,
    input_images=None,
    system_prompt=None,
    history_messages=None,
    max_new_tokens=1024,
):
    _load_model()

    messages = _build_messages(prompt, input_images, system_prompt, history_messages)

    with _gen_lock:
        try:
            return _generate_once(messages, max_new_tokens)
        except torch.cuda.OutOfMemoryError:
            logger.warning("OOM detected. Retrying without images and shorter prompt.")
            _free_gpu()
            messages = _build_messages(prompt, None, system_prompt, history_messages)
            try:
                return _generate_once(messages, min(max_new_tokens, 512))
            except torch.cuda.OutOfMemoryError:
                _free_gpu()
                raise


async def qwen_gpt_4o_mini_complete(
    prompt,
    input_images=None,
    system_prompt=None,
    history_messages=None,
    keyword_extraction=False,
    **kwargs,
) -> str:
    token_tracker = kwargs.pop("token_tracker", None)

    kwargs.pop("response_format", None)
    kwargs.pop("hashing_kv", None)
    kwargs.pop("stream", None)

    max_new_tokens = int(kwargs.pop("max_tokens", 256 if keyword_extraction else 1024))

    try:
        text, prompt_tokens, completion_tokens = await asyncio.to_thread(
            _generate_sync,
            prompt,
            input_images,
            system_prompt,
            history_messages,
            max_new_tokens,
        )
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

    if token_tracker is not None and hasattr(token_tracker, "add_usage"):
        try:
            token_tracker.add_usage(
                {
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": completion_tokens,
                    "total_tokens": prompt_tokens + completion_tokens,
                }
            )
        except Exception:
            pass

    return text

Writing /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/qwen_llm.py


In [8]:
######################################################
# Script OCR bằng Qwen3 VL
######################################################

In [9]:
%%writefile /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/ocr_pages.py
import argparse
import asyncio
import json
import pathlib
import sys

sys.path.append(str(pathlib.Path(__file__).resolve().parent))
from qwen_llm import qwen_gpt_4o_mini_complete

SYSTEM_PROMPT = "Bạn là trợ lý OCR tài liệu tiếng Việt, chuyên trích xuất văn bản từ ảnh trang sách, đề bài, bảng biểu."

OCR_PROMPT = """Hãy trích xuất nguyên văn tiếng Việt từ ảnh tài liệu này.

Yêu cầu:
- Giữ nguyên thứ tự nội dung.
- Nếu có bảng, hãy mô tả dạng văn bản thuần.
- Không thêm bình luận, không giải thích.
- Nếu không đọc được, trả về đúng: Không đọc được văn bản.
"""


async def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True)
    ap.add_argument("--output", required=True)
    ap.add_argument("--max-chars", type=int, default=3000)
    args = ap.parse_args()

    with open(args.input, "r", encoding="utf-8") as f:
        pages = json.load(f)

    for page_idx, page in pages.items():
        if str(page.get("text", "")).strip():
            continue

        img_path = page.get("page_image")
        try:
            text = await qwen_gpt_4o_mini_complete(
                OCR_PROMPT,
                input_images=[img_path],
                system_prompt=SYSTEM_PROMPT,
                max_tokens=1024,
            )
        except Exception as e:
            text = f"Không đọc được văn bản. Lỗi: {e}"

        if text.startswith("ERROR:"):
            text = "Không đọc được văn bản."

        page["text"] = text[: args.max_chars]
        print(f"OCR page {page_idx}: {len(page['text'])} chars")

    with open(args.output, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    print(f"Saved OCR pages to {args.output}")


if __name__ == "__main__":
    asyncio.run(main())

Writing /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/ocr_pages.py


In [10]:
######################################################
# Create baseline no-RAG -> Qwen3 VL trả lời không RAG
######################################################

In [11]:
%%writefile /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/baseline_direct.py
import argparse
import asyncio
import json
import pathlib
import sys

sys.path.append(str(pathlib.Path(__file__).resolve().parent))
from qwen_llm import qwen_gpt_4o_mini_complete

SYSTEM_PROMPT = "Bạn là trợ lý trả lời câu hỏi dựa trên ảnh tài liệu tiếng Việt. Trả lời ngắn gọn, chính xác, bằng tiếng Việt."

PROMPT_TEMPLATE = """Dựa vào ảnh tài liệu này, hãy trả lời câu hỏi sau một cách chính xác và ngắn gọn bằng tiếng Việt:

Câu hỏi: {question}

Trả lời:"""


async def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--eval-items", required=True)
    ap.add_argument("--pages", required=True)
    ap.add_argument("--output", required=True)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    with open(args.eval_items, "r", encoding="utf-8") as f:
        eval_items = [json.loads(line) for line in f if line.strip()]

    with open(args.pages, "r", encoding="utf-8") as f:
        pages = json.load(f)

    # resume support
    done_ids = set()
    if args.resume and pathlib.Path(args.output).exists():
        with open(args.output, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        done_ids.add(json.loads(line)["index"])
                    except Exception:
                        pass

    mode = "a" if args.resume else "w"
    with open(args.output, mode, encoding="utf-8") as fout:
        for item in eval_items:
            if item["index"] in done_ids:
                continue

            page_idx = str(item.get("page_idx", ""))
            img = pages.get(page_idx, {}).get("page_image")

            try:
                ans = await qwen_gpt_4o_mini_complete(
                    PROMPT_TEMPLATE.format(question=item["question"]),
                    input_images=[img] if img else None,
                    system_prompt=SYSTEM_PROMPT,
                    max_tokens=512,
                )
                err = None
            except Exception as e:
                ans = ""
                err = str(e)

            rec = {
                "index": item["index"],
                "question": item["question"],
                "answer": ans,
                "error": err,
            }
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fout.flush()
            print(f"[baseline] done index={item['index']}")

    print(f"Saved baseline results to {args.output}")


if __name__ == "__main__":
    asyncio.run(main())

Writing /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/baseline_direct.py


In [12]:
######################################################
# Script LLM-judge (Qwen3) -> Evaluate Question-Answer QA 
######################################################

In [13]:
%%writefile /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/judge_qwen.py
import argparse
import asyncio
import json
import pathlib
import random
import re
import sys

sys.path.append(str(pathlib.Path(__file__).resolve().parent))
from qwen_llm import qwen_gpt_4o_mini_complete

SYSTEM_PROMPT = "Bạn là giám khảo đánh giá câu trả lời QA tài liệu tiếng Việt."

JUDGE_PROMPT = """Hãy đánh giá câu trả lời của mô hình so với câu trả lời mẫu.

Câu hỏi:
{question}

Câu trả lời mẫu:
{gold_answer}

Câu trả lời mô hình:
{pred_answer}

Yêu cầu:
- Trả về đúng một JSON hợp lệ, không markdown.
- JSON gồm 2 trường: "verdict" và "reason".
- "verdict" chỉ được là "YES" hoặc "NO".
- "YES" nếu câu trả lời mô hình đúng về ngữ nghĩa và đủ ý so với câu trả lời mẫu.
- "NO" nếu sai, thiếu ý quan trọng, hoặc không liên quan.

JSON:
"""


def parse_verdict(resp: str):
    if not resp:
        return "NO", "empty response"

    m = re.search(r"\{.*\}", resp, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            verdict = str(obj.get("verdict", "")).upper().strip()
            reason = str(obj.get("reason", ""))
            if verdict in ["YES", "NO"]:
                return verdict, reason
            if "YES" in verdict:
                return "YES", reason
            if "NO" in verdict:
                return "NO", reason
        except Exception:
            pass

    upper = resp.upper()
    if "YES" in upper:
        return "YES", resp[:300]
    return "NO", resp[:300]


async def judge_item(sem, item):
    async with sem:
        prompt = JUDGE_PROMPT.format(
            question=item["question"],
            gold_answer=item["gold_answer"],
            pred_answer=item["pred_answer"],
        )
        try:
            resp = await qwen_gpt_4o_mini_complete(
                prompt,
                system_prompt=SYSTEM_PROMPT,
                max_tokens=256,
            )
        except Exception as e:
            resp = f"ERROR: {e}"

        verdict, reason = parse_verdict(resp)
        return {
            "index": item["index"],
            "question": item["question"],
            "verdict": verdict,
            "reason": reason,
            "raw_judge_response": resp,
        }


async def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--eval-items", required=True)
    ap.add_argument("--results", required=True)
    ap.add_argument("--output", required=True)
    ap.add_argument("--sample", type=int, default=20)
    args = ap.parse_args()

    with open(args.eval_items, "r", encoding="utf-8") as f:
        eval_items = [json.loads(line) for line in f if line.strip()]

    preds = {}
    with open(args.results, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            preds[rec["index"]] = rec.get("answer", "") if not rec.get("error") else ""

    judge_inputs = []
    for item in eval_items:
        idx = item["index"]
        if idx in preds:
            judge_inputs.append(
                {
                    "index": idx,
                    "question": item["question"],
                    "gold_answer": item["gold_answer"],
                    "pred_answer": preds[idx],
                }
            )

    random.seed(42)
    if args.sample and args.sample < len(judge_inputs):
        judge_inputs = random.sample(judge_inputs, args.sample)

    sem = asyncio.Semaphore(1)
    tasks = [judge_item(sem, item) for item in judge_inputs]
    results = await asyncio.gather(*tasks)

    with open(args.output, "w", encoding="utf-8") as f:
        for rec in results:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Saved judge results to {args.output}")


if __name__ == "__main__":
    asyncio.run(main())

Writing /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/judge_qwen.py


In [14]:
######################################################
# Patch code MegaRAG dùng Qwen3-VL
######################################################

In [15]:
import re
import pathlib

repo = pathlib.Path("/kaggle/working/megarag_vdoc/MegaRAG")

# 1) Sửa prompt two-step -> xóa trả lời English
prompt_path = repo / "megarag" / "prompt.py"
if prompt_path.exists():
    s = prompt_path.read_text(encoding="utf-8")
    s = s.replace(
        "Please respond in English.",
        "Please respond in the same language as the user's question."
    )
    prompt_path.write_text(s, encoding="utf-8")

# 2) dùng Qwen local
scripts = [
    "egs/utils/construct_mmkg.py",
    "egs/utils/query_mmkg.py",
]

for rel in scripts:
    p = repo / rel
    s = p.read_text(encoding="utf-8")

    if not any(line.strip().startswith("import os") for line in s.splitlines()[:40]):
        s = "import os\n" + s

    # thay import
    s = re.sub(
        r"from\s+megarag\.llms\.openai\s+import\s*\(\s*gpt_4o_mini_complete\s*,?\s*\)",
        "from qwen_llm import qwen_gpt_4o_mini_complete as gpt_4o_mini_complete",
        s,
        flags=re.S,
    )

    # thay import 1 line
    s = re.sub(
        r"from\s+megarag\.llms\.openai\s+import\s+gpt_4o_mini_complete",
        "from qwen_llm import qwen_gpt_4o_mini_complete as gpt_4o_mini_complete",
        s,
    )

    # chọn device cho embedding model
    s = re.sub(
        r'device\s*=\s*"cuda" if torch\.cuda\.is_available\(\) else "cpu"',
        'device = os.environ.get("EMBED_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")',
        s,
    )

    s = re.sub(
        r"torch_dtype=torch\.float16 if torch\.cuda\.is_available\(\) else torch\.float32",
        'torch_dtype=torch.float16 if str(device).startswith("cuda") else torch.float32',
        s,
    )

    s = re.sub(
        r'device_map="cuda" if torch\.cuda\.is_available\(\) else None',
        'device_map=device if str(device).startswith("cuda") else None',
        s,
    )

    p.write_text(s, encoding="utf-8")

print("Patched MegaRAG scripts.")

Patched MegaRAG scripts.


In [16]:
######################################################
# down dataset preview + prepare page QA
######################################################

In [17]:
import os
import io
import json
import shutil
import hashlib
import pathlib
import random

from PIL import Image
from datasets import load_dataset

Image.MAX_IMAGE_PIXELS = None

random.seed(42)

TMP = pathlib.Path(os.environ["TMP_DIR"])
IMG_DIR = TMP / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "trannhiem/TranNhiem-Vietnamese-DocumentImage-Reasoning"
CONFIG_NAME = "preview"  # 300 mẫu; muốn full thì đổi thành "full" nhưng rất nặng
SPLIT = "train"

SAMPLE_PAGES = int(os.environ.get("SAMPLE_PAGES", "40"))
MAX_QUESTIONS_PER_PAGE = int(os.environ.get("MAX_QUESTIONS_PER_PAGE", "4"))
MAX_EVAL_QUESTIONS = int(os.environ.get("MAX_EVAL_QUESTIONS", "120"))

print("Loading dataset preview...")
ds = load_dataset(
    DATASET_ID,
    CONFIG_NAME,
    split=SPLIT,
    cache_dir=os.environ["HF_DATASETS_CACHE"],
)

print(f"Loaded {len(ds)} rows from {DATASET_ID} / {CONFIG_NAME}")


def save_image(img, path):
    path = pathlib.Path(path)
    if isinstance(img, Image.Image):
        img.convert("RGB").save(path, quality=90)
    elif isinstance(img, dict):
        if img.get("bytes") is not None:
            Image.open(io.BytesIO(img["bytes"])).convert("RGB").save(path, quality=90)
        elif img.get("path") and pathlib.Path(img["path"]).exists():
            shutil.copy(img["path"], path)
        elif img.get("src"):
            import requests
            r = requests.get(img["src"], timeout=30)
            Image.open(io.BytesIO(r.content)).convert("RGB").save(path, quality=90)
        else:
            raise ValueError("Cannot interpret image dict")
    elif isinstance(img, (str, pathlib.Path)):
        p = pathlib.Path(img)
        if p.exists():
            shutil.copy(p, path)
        else:
            import requests
            r = requests.get(img, timeout=30)
            Image.open(io.BytesIO(r.content)).convert("RGB").save(path, quality=90)
    else:
        raise ValueError(f"Unknown image type: {type(img)}")


# Questionn theo page id
groups = {}

for i, row in enumerate(ds):
    img = row.get("image", None)
    if img is None:
        continue

    pid = str(row.get("id", i))
    q = row.get("question", "")
    a = row.get("model_answer", "")

    if not q or not a:
        continue

    if pid not in groups:
        groups[pid] = {
            "image": img,
            "questions": [],
        }

    groups[pid]["questions"].append(
        {
            "question": q,
            "model_answer": a,
            "model_reasoning": row.get("model_reasoning", ""),
        }
    )

print(f"Unique page ids: {len(groups)}")

# page nhiều QA 
page_ids = list(groups.keys())
page_ids.sort(key=lambda pid: len(groups[pid]["questions"]), reverse=True)

selected_page_ids = page_ids[:SAMPLE_PAGES]
print(f"Selected {len(selected_page_ids)} pages: {selected_page_ids}")

pages_content = {}
eval_items = []
qid = 0

for page_idx, pid in enumerate(selected_page_ids):
    img_path = IMG_DIR / f"page_{page_idx:03d}.jpg"
    save_image(groups[pid]["image"], img_path)

    pages_content[str(page_idx)] = {
        "text": "",  # OCR / placeholder
        "page_image": str(img_path),
        "figure_images": [],
        "page_id": pid,
    }

    qs = groups[pid]["questions"][:MAX_QUESTIONS_PER_PAGE]
    for q in qs:
        if len(eval_items) >= MAX_EVAL_QUESTIONS:
            break
        eval_items.append(
            {
                "index": qid,
                "page_idx": page_idx,
                "page_id": pid,
                "question": q["question"],
                "gold_answer": q["model_answer"],
            }
        )
        qid += 1

if len(eval_items) == 0:
    raise RuntimeError("No evaluation questions were created. Increase SAMPLE_PAGES.")

pre_ocr_path = TMP / "pages_content_preocr.json"
with open(pre_ocr_path, "w", encoding="utf-8") as f:
    json.dump(pages_content, f, ensure_ascii=False, indent=2)

with open(TMP / "eval_items.jsonl", "w", encoding="utf-8") as f:
    for item in eval_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved pre-OCR pages: {pre_ocr_path}")
print(f"Saved eval items: {TMP / 'eval_items.jsonl'}")
print(f"Number of eval questions: {len(eval_items)}")

Loading dataset preview...


README.md: 0.00B [00:00, ?B/s]

preview/preview-000.parquet:   0%|          | 0.00/99.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Loaded 300 rows from trannhiem/TranNhiem-Vietnamese-DocumentImage-Reasoning / preview
Unique page ids: 100
Selected 30 pages: ['96220', '66496', '45970', '85365', '104466', '95427', '34272', '86693', '103525', '95432', '16869', '76605', '77216', '94830', '2777', '29613', '32348', '56933', '79772', '104308', '104094', '28629', '27421', '5689', '79402', '65877', '77792', '36286', '17596', '16725']
Saved pre-OCR pages: /kaggle/tmp/megarag_vdoc/pages_content_preocr.json
Saved eval items: /kaggle/tmp/megarag_vdoc/eval_items.jsonl
Number of eval questions: 90


In [18]:
######################################################
# OCR -> OCR ảnh page / tạo placeholder
######################################################

In [19]:
import os
import json
import pathlib

TMP = pathlib.Path(os.environ["TMP_DIR"])
repo = pathlib.Path("/kaggle/working/megarag_vdoc/MegaRAG")

pre_ocr = TMP / "pages_content_preocr.json"
final_pages = TMP / "pages_content.json"

USE_OCR = os.environ.get("USE_OCR", "1") == "1"

if USE_OCR:
    print("Running OCR with Qwen3-VL... This may take several minutes.")
    !cd {repo} && \
      CUDA_VISIBLE_DEVICES=0 \
      HF_HOME=/kaggle/tmp/megarag_vdoc/hf \
      TRANSFORMERS_CACHE=/kaggle/tmp/megarag_vdoc/hf/models \
      python egs/utils/ocr_pages.py \
        --input {pre_ocr} \
        --output {final_pages} \
        --max-chars 3000
else:
    print("USE_OCR=0 -> using placeholder text.")
    with open(pre_ocr, "r", encoding="utf-8") as f:
        pages = json.load(f)

    for _, v in pages.items():
        if not str(v.get("text", "")).strip():
            v["text"] = "Ảnh trang tài liệu tiếng Việt. Vui lòng xem hình."

    with open(final_pages, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

print(f"Final pages_content path: {final_pages}")

Running OCR with Qwen3-VL... This may take several minutes.
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
[qwen_llm] Detected 1 GPU(s): ['Tesla T4']
`torch_dtype` is deprecated! Use `dtype` instead!
config.json: 2.28kB [00:00, 10.9MB/s]
model.safetensors.index.json: 209kB [00:00, 220MB/s]
Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
model-00001-of-00002.safetensors:   0%|             | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|             | 0.00/2.23G [00:00<?, ?B/s]
model-00001-of-00002.safetensors:   0%|             | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|             | 0.00/2.23G [00:00<?, ?B/s]
model-00001-of-00002.safetensors:   0%|             | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|             | 0.0

In [20]:
######################################################
# Tạo recipe egs/vdoc + config addon_params.yaml
######################################################

In [21]:
# chỉnh sang tiếng Việt + giảm batch
import os
import shutil
import pathlib

repo = pathlib.Path("/kaggle/working/megarag_vdoc/MegaRAG")
recipe = repo / "egs" / "vdoc"

# Xoá recipe cũ nếu có
if recipe.exists():
    shutil.rmtree(recipe)

# Copy template
shutil.copytree(repo / "egs" / ".template", recipe)

# Tạo thư mục data/exp
(recipe / "data").mkdir(exist_ok=True)
(recipe / "exp").mkdir(exist_ok=True)

addon_params = """example_number: 1
language: Vietnamese

entity_types:
  - person
  - organization
  - location
  - event
  - concept
  - literary_work
  - table
  - figure
  - date
  - mathematical_expression
  - document
  - law

insert_batch_size: 1
entity_extract_max_gleaning: 0
entity_refine_max_times: 0

refine_subgraph_top_k: 20
refine_subgraph_max_token_for_global_context: 3000
refine_subgraph_max_token_for_local_context: 3000
refine_subgraph_max_token_for_text_unit: 3000

chunk_top_k: 4
embed_parallel_limit: 1
llm_model_max_async: 1
"""

conf_path = recipe / "conf" / "addon_params.yaml"
conf_path.parent.mkdir(parents=True, exist_ok=True)
conf_path.write_text(addon_params, encoding="utf-8")

print(f"Created recipe at {recipe}")
print(f"Config written to {conf_path}")

Created recipe at /kaggle/working/megarag_vdoc/MegaRAG/egs/vdoc
Config written to /kaggle/working/megarag_vdoc/MegaRAG/egs/vdoc/conf/addon_params.yaml


In [22]:
######################################################
# Tạo query -> tạo query từ câu hỏi đánh giá
######################################################

In [23]:
import os
import json
import pathlib

TMP = pathlib.Path(os.environ["TMP_DIR"])
OUT = pathlib.Path(os.environ["OUT_DIR"])
OUT.mkdir(parents=True, exist_ok=True)

eval_items_path = TMP / "eval_items.jsonl"
queries_path = TMP / "queries.jsonl"
eval_items_out = OUT / "eval_items.jsonl"

with open(eval_items_path, "r", encoding="utf-8") as f:
    eval_items = [json.loads(line) for line in f if line.strip()]

with open(queries_path, "w", encoding="utf-8") as f:
    for item in eval_items:
        f.write(json.dumps({"question": item["question"]}, ensure_ascii=False) + "\n")

# Copy eval_items ra output để đánh giá
with open(eval_items_out, "w", encoding="utf-8") as f:
    for item in eval_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved queries to {queries_path}")
print(f"Saved eval items to {eval_items_out}")
print(f"Total queries: {len(eval_items)}")

Saved queries to /kaggle/tmp/megarag_vdoc/queries.jsonl
Saved eval items to /kaggle/working/megarag_outputs/eval_items.jsonl
Total queries: 90


In [24]:
######################################################
#  load GME file gme_compat.py
######################################################

In [25]:
# chỉnh load GME tương thích với transformers 4.57

In [26]:
%%writefile /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/gme_compat.py
from __future__ import annotations

import base64
import logging
import math
import os
from io import BytesIO
from typing import Dict, List, Optional

import requests
import torch
from PIL import Image
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import AutoConfig, AutoProcessor, AutoModelForVision2Seq

MODEL_NAME = "Alibaba-NLP/gme-Qwen2-VL-2B-Instruct"

IMAGE_FACTOR = 28
MIN_PIXELS = 4 * 28 * 28
# Giảm max pixels để an toàn hơn trên Kaggle
MAX_PIXELS = 1280 * 28 * 28
MAX_RATIO = 200


def round_by_factor(number: int, factor: int) -> int:
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    return math.floor(number / factor) * factor


def smart_resize(
    height: int,
    width: int,
    factor: int = IMAGE_FACTOR,
    min_pixels: int = MIN_PIXELS,
    max_pixels: int = MAX_PIXELS,
) -> tuple[int, int]:
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))

    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)

    if max(h_bar, w_bar) / min(h_bar, w_bar) > MAX_RATIO:
        logging.warning(
            f"Absolute aspect ratio must be smaller than {MAX_RATIO}, "
            f"got {max(h_bar, w_bar) / min(h_bar, w_bar)}"
        )
        if h_bar > w_bar:
            h_bar = w_bar * MAX_RATIO
        else:
            w_bar = h_bar * MAX_RATIO

    return h_bar, w_bar


def fetch_image(image, size_factor: int = IMAGE_FACTOR) -> Image.Image:
    image_obj = None

    if isinstance(image, Image.Image):
        image_obj = image
    elif isinstance(image, str):
        if image.startswith("http://") or image.startswith("https://"):
            image_obj = Image.open(requests.get(image, stream=True, timeout=30).raw)
        elif image.startswith("file://"):
            image_obj = Image.open(image[7:])
        elif image.startswith("data:image"):
            if "base64," in image:
                _, base64_data = image.split("base64,", 1)
                data = base64.b64decode(base64_data)
                image_obj = Image.open(BytesIO(data))
        elif os.path.exists(image):
            image_obj = Image.open(image)

    if image_obj is None:
        raise ValueError(f"Unrecognized image input: {image}")

    image = image_obj.convert("RGB")
    width, height = image.size

    resized_height, resized_width = smart_resize(
        height,
        width,
        factor=size_factor,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )
    image = image.resize((resized_width, resized_height))
    return image


def custom_collate_fn(batch):
    return batch


class GmeQwen2VLCompat:
    """
    wrapper tương thích transformers 4.57 cho GME-Qwen2-VL-2B-Instruct
    """

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        device: Optional[str] = None,
        dtype: Optional[torch.dtype] = None,
        min_image_tokens: int = 256,
        max_image_tokens: int = 1280,
        max_length: int = 1800,
        **kwargs,
    ) -> None:
        if device is None:
            device = os.environ.get(
                "EMBED_DEVICE",
                "cuda" if torch.cuda.is_available() else "cpu",
            )

        self.device = device

        if dtype is None:
            dtype = torch.float16 if str(device).startswith("cuda") else torch.float32

        # Force Qwen2-VL config, tránh remote GME config
        try:
            from transformers.models.qwen2_vl.configuration_qwen2_vl import Qwen2VLConfig
            config = Qwen2VLConfig.from_pretrained(
                model_name,
                trust_remote_code=False,
            )
        except Exception:
            config = AutoConfig.from_pretrained(
                model_name,
                trust_remote_code=False,
            )

        config.model_type = "qwen2_vl"
        config.architectures = ["Qwen2VLForConditionalGeneration"]

        if hasattr(config, "auto_map"):
            try:
                delattr(config, "auto_map")
            except Exception:
                pass

        if not getattr(config, "_name_or_path", None):
            config._name_or_path = model_name

        self.base = AutoModelForVision2Seq.from_pretrained(
            model_name,
            config=config,
            torch_dtype=dtype,
            trust_remote_code=False,
            ignore_mismatched_sizes=True,
            low_cpu_mem_usage=True,
        )

        self.base.eval()
        self.base.to(self.device)

        self.normalize = True
        self.max_length = max_length

        min_pixels = min_image_tokens * 28 * 28
        max_pixels = max_image_tokens * 28 * 28

        self.processor = AutoProcessor.from_pretrained(
            model_name,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
            trust_remote_code=False,
        )
        self.processor.tokenizer.padding_side = "right"
        self.default_instruction = "You are a helpful assistant."
        self.sep = " "

    def _get_input_embeddings(self):
        # Ưu tiên API mới: get_input_embeddings()
        if hasattr(self.base, "get_input_embeddings"):
            try:
                emb = self.base.get_input_embeddings()
                if emb is not None:
                    return emb
            except Exception:
                pass

        backbone = getattr(self.base, "model", None)

        if backbone is not None and hasattr(backbone, "get_input_embeddings"):
            try:
                emb = backbone.get_input_embeddings()
                if emb is not None:
                    return emb
            except Exception:
                pass

        # base_model.language_model.embed_tokens
        if backbone is not None and hasattr(backbone, "language_model"):
            return backbone.language_model.get_input_embeddings()

        raise AttributeError("Cannot locate input embeddings for GME wrapper.")

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values=None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        pixel_values: Optional[torch.Tensor] = None,
        image_grid_thw: Optional[torch.LongTensor] = None,
        pooling_mask: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> torch.Tensor:
        backbone = getattr(self.base, "model", self.base)

        if inputs_embeds is None:
            embed_layer = self._get_input_embeddings()
            inputs_embeds = embed_layer(input_ids)

            if pixel_values is not None:
                visual = getattr(self.base, "visual", getattr(backbone, "visual", None))
                if visual is None:
                    raise AttributeError("Cannot locate visual encoder for GME wrapper.")

                pixel_values = pixel_values.type(visual.get_dtype())
                image_embeds = visual(pixel_values, grid_thw=image_grid_thw)
                image_embeds = image_embeds.to(inputs_embeds.device)

                image_mask = input_ids == self.base.config.image_token_id
                inputs_embeds[image_mask] = image_embeds.to(inputs_embeds.dtype)

            if attention_mask is not None:
                attention_mask = attention_mask.to(inputs_embeds.device)

        outputs = backbone(
            input_ids=None,
            position_ids=position_ids,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
        )

        hidden_states = outputs.last_hidden_state
        pooling_mask = attention_mask if pooling_mask is None else pooling_mask

        left_padding = pooling_mask[:, -1].sum() == pooling_mask.shape[0]

        if left_padding:
            embeddings = hidden_states[:, -1]
        else:
            sequence_lengths = pooling_mask.sum(dim=1) - 1
            batch_size = hidden_states.shape[0]
            embeddings = hidden_states[
                torch.arange(batch_size, device=hidden_states.device),
                sequence_lengths,
            ]

        if self.normalize:
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings.contiguous()

    def embed(self, texts: List[str], images: List, is_query=True, instruction=None, **kwargs):
        self.base.to(self.device)
        input_texts, input_images = [], []

        for t, i in zip(texts, images):
            if not is_query or instruction is None:
                instruction = self.default_instruction

            input_str = ""

            if i is None:
                input_images = None
            else:
                input_str += "<|vision_start|><|image_pad|><|vision_end|>"
                i = fetch_image(i)
                input_images.append(i)

            if t is not None:
                input_str += t

            msg = (
                f"<|im_start|>system\n{instruction}<|im_end|>\n"
                f"<|im_start|>user\n{input_str}<|im_end|>\n"
                f"<|im_start|>assistant\n<|endoftext|>"
            )
            input_texts.append(msg)

        inputs = self.processor(
            text=input_texts,
            images=input_images,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        inputs = {
            k: (v.to(self.device) if hasattr(v, "to") else v)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            embeddings = self.forward(**inputs)

        return embeddings

    def encode(self, sentences: List[str], *, prompt_name=None, **kwargs):
        return self.get_fused_embeddings(texts=sentences, prompt_name=prompt_name, **kwargs)

    def encode_queries(self, queries: List[str], **kwargs):
        return self.encode(queries, **kwargs)

    def encode_corpus(self, corpus: List[Dict[str, str]], **kwargs):
        if isinstance(corpus, dict):
            sentences = [
                (corpus["title"][i] + self.sep + corpus["text"][i]).strip()
                if "title" in corpus
                else corpus["text"][i].strip()
                for i in range(len(corpus["text"]))
            ]
        else:
            sentences = [
                (doc["title"] + self.sep + doc["text"]).strip()
                if "title" in doc
                else doc["text"].strip()
                for doc in corpus
            ]
        return self.encode(sentences, is_query=False, **kwargs)

    def get_image_embeddings(self, images, **kwargs):
        return self.get_fused_embeddings(images=images, **kwargs)

    def get_text_embeddings(self, texts: List[str], **kwargs):
        return self.get_fused_embeddings(texts=texts, **kwargs)

    def get_fused_embeddings(
        self,
        texts: Optional[List[str]] = None,
        images=None,
        **kwargs,
    ):
        if isinstance(images, DataLoader):
            image_loader = images
            batch_size = image_loader.batch_size
            if hasattr(image_loader, "dataset") and hasattr(image_loader.dataset, "transform"):
                image_loader.dataset.transform = None
        else:
            # set batch_size nhỏ 
            batch_size = kwargs.pop("batch_size", 1)

            if images is None:
                image_loader = None
            else:
                image_loader = DataLoader(
                    images,
                    batch_size=batch_size,
                    shuffle=False,
                    collate_fn=custom_collate_fn,
                    num_workers=0,
                )

        if texts is None:
            assert image_loader is not None
            n_batch = len(image_loader)
        else:
            n_batch = len(texts) // batch_size + int(len(texts) % batch_size > 0)
            image_loader = image_loader or [None] * n_batch

        all_embeddings = []
        none_batch = [None] * batch_size
        show_progress_bar = kwargs.pop("show_progress_bar", False)

        pbar = tqdm(
            total=n_batch,
            disable=not show_progress_bar,
            mininterval=1,
            miniters=10,
            desc="GME encode",
        )

        for n, img_batch in zip(range(0, n_batch * batch_size, batch_size), image_loader):
            text_batch = none_batch if texts is None else texts[n : n + batch_size]
            img_batch = none_batch if img_batch is None else img_batch

            embeddings = self.embed(
                texts=text_batch,
                images=img_batch,
                **kwargs,
            )

            pbar.update(1)
            all_embeddings.append(embeddings.cpu())

        pbar.close()
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return all_embeddings


def load_gme_model():
    """
    Hàm được gọi thay cho initialize_model() trong construct_mmkg.py / query_mmkg.py.
    """
    device = os.environ.get(
        "EMBED_DEVICE",
        "cuda" if torch.cuda.is_available() else "cpu",
    )

    try:
        return GmeQwen2VLCompat(device=device)
    except Exception as e:
        print(f"[gme_compat] Primary loader failed: {e}")
        print("[gme_compat] Fallback: patch require_version and load remote GME.")

        # Fallback nếu loader chính thất bại
        import transformers.utils.versions as versions
        versions.require_version = lambda *args, **kwargs: None

        from transformers import AutoModel

        config = AutoConfig.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True,
        )

        if not getattr(config, "_name_or_path", None):
            config._name_or_path = MODEL_NAME

        dtype = torch.float16 if str(device).startswith("cuda") else torch.float32

        model = AutoModel.from_pretrained(
            MODEL_NAME,
            config=config,
            torch_dtype=dtype,
            device_map=device if str(device).startswith("cuda") else None,
            trust_remote_code=True,
        )

        model = model.to(device)
        model.eval()
        return model

Writing /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/gme_compat.py


In [27]:
######################################################
# patch initialize_model MegaRAG
######################################################

In [28]:
# thay thế initialize_model() trong construct_mmkg.py và query_mmkg.py để 
# dùng loader ở trên

In [29]:
import pathlib
import re

repo = pathlib.Path("/kaggle/working/megarag_vdoc/MegaRAG")

new_initialize = '''def initialize_model():
    from gme_compat import load_gme_model
    return load_gme_model()


'''

files = [
    repo / "egs" / "utils" / "construct_mmkg.py",
    repo / "egs" / "utils" / "query_mmkg.py",
]

for path in files:
    s = path.read_text(encoding="utf-8")

    # Thay toàn bộ hàm initialize_model cũ
    s = re.sub(
        r"def initialize_model\(\):.*?(?=\ndef |\nclass |\Z)",
        new_initialize,
        s,
        count=1,
        flags=re.S,
    )

    path.write_text(s, encoding="utf-8")
    print(f"Patched {path}")

print("Done patching initialize_model.")

Patched /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/construct_mmkg.py
Patched /kaggle/working/megarag_vdoc/MegaRAG/egs/utils/query_mmkg.py
Done patching initialize_model.


In [30]:
######################################################
# build MMKG bằng MegaRAG + Qwen3-VL
######################################################

In [31]:
%%bash
set -e

export HF_HOME=/kaggle/tmp/megarag_vdoc/hf
export HF_DATASETS_CACHE=/kaggle/tmp/megarag_vdoc/hf/datasets
export TRANSFORMERS_CACHE=/kaggle/tmp/megarag_vdoc/hf/models
export CUDA_VISIBLE_DEVICES=0
export EMBED_DEVICE=cpu
export PYTHONPATH=/kaggle/working/megarag_vdoc/MegaRAG

cd /kaggle/working/megarag_vdoc/MegaRAG

# Xoá index cũ nếu có
rm -rf egs/vdoc/exp

python egs/utils/construct_mmkg.py \
  --config-file egs/vdoc/conf/addon_params.yaml \
  --working-dir egs/vdoc/exp/vdoc \
  --input-dir /kaggle/tmp/megarag_vdoc/pages_content.json \
  --max-retries 1 \
  --retry-delay 5

echo "Built MMKG."

[qwen_llm] Detected 1 GPU(s): ['Tesla T4']
Final Token Usage: LLM call count: 60, Prompt tokens: 172703, Completion tokens: 53790, Total tokens: 226493.
Built MMKG.


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 30.52it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Rerank is enabled but no rera

In [32]:
######################################################
# query MegaRAG trên câu hỏi tiếng Việt
######################################################

In [33]:
%%bash
set -e

export HF_HOME=/kaggle/tmp/megarag_vdoc/hf
export HF_DATASETS_CACHE=/kaggle/tmp/megarag_vdoc/hf/datasets
export TRANSFORMERS_CACHE=/kaggle/tmp/megarag_vdoc/hf/models
export CUDA_VISIBLE_DEVICES=0
export EMBED_DEVICE=cpu
export PYTHONPATH=/kaggle/working/megarag_vdoc/MegaRAG

cd /kaggle/working/megarag_vdoc/MegaRAG

mkdir -p /kaggle/working/megarag_outputs/results

python egs/utils/query_mmkg.py \
  --config-file egs/vdoc/conf/addon_params.yaml \
  --working-dir egs/vdoc/exp/vdoc \
  --input-queries /kaggle/tmp/megarag_vdoc/queries.jsonl \
  --output-file /kaggle/working/megarag_outputs/results/results.jsonl \
  --output-format jsonl \
  --concurrency 1 \
  --max-retries 1 \
  --retry-delay 5

echo "Done querying."

[qwen_llm] Detected 1 GPU(s): ['Tesla T4']
[0] (118.22s) Theo văn bản, những tác phẩm được liệt kê là của tác giả **Vũ Đình Long** bao gồm:

1. **Chén thuốc độc** (năm 1921)  
2. **Toả án lương tâm** (năm 1923)  
3. **Đàn bà mới** (năm 1944)  
4. **Tổ quốc trên hết** (năm 1949, phóng tác)  
5. **Gia tài** (năm 1958, phóng tác)

**Bổ sung thông tin từ văn bản**:  
- *Gia tài* là tác phẩm do Vũ Đình Long phóng tác từ vở hài kịch *Lê-ga-tê Uy-ni-véc-xen* (Légataire Universel) của Ro-nha (Regnard).

**Lưu ý**:  
- Văn bản liệt kê tổng cộng **5 tác phẩm** của Vũ Đình Long, trong đó có hai tác phẩm được ghi rõ năm sáng tác (1921 và 1923), ba tác phẩm khác được ghi rõ năm phóng tác (1944, 1949, 1958).  
- Các tác phẩm này đều là những đóng góp quan trọng của ông trong lĩnh vực kịch Việt Nam, đặc biệt là trong giai đoạn đầu thế kỷ XX và những năm sau đó, thể hiện sự phát triển và đổi mới trong nghệ thuật sân khấu đương thời.

*(Tổng hợp từ cả hai nguồn: Knowledge Graph chỉ liệt kê 2 tác phẩm đ

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 34.42it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Rerank is enabled but no rera

In [34]:
######################################################
# baseline no-RAG
######################################################

In [35]:
%%bash
if [ "$RUN_BASELINE" != "1" ]; then
  echo "RUN_BASELINE != 1, skip."
  exit 0
fi

export HF_HOME=/kaggle/tmp/megarag_vdoc/hf
export TRANSFORMERS_CACHE=/kaggle/tmp/megarag_vdoc/hf/models
export CUDA_VISIBLE_DEVICES=0,1
export PYTHONPATH=/kaggle/working/megarag_vdoc/MegaRAG

cd /kaggle/working/megarag_vdoc/MegaRAG

python egs/utils/baseline_direct.py \
  --eval-items /kaggle/working/megarag_outputs/eval_items.jsonl \
  --pages /kaggle/tmp/megarag_vdoc/pages_content.json \
  --output /kaggle/working/megarag_outputs/results/baseline_results.jsonl \
  --resume

echo "Done baseline."

[qwen_llm] Detected 2 GPU(s): ['Tesla T4', 'Tesla T4']
[baseline] done index=0
[baseline] done index=1
[baseline] done index=2
[baseline] done index=3
[baseline] done index=4
[baseline] done index=5
[baseline] done index=6
[baseline] done index=7
[baseline] done index=8
[baseline] done index=9
[baseline] done index=10
[baseline] done index=11
[baseline] done index=12
[baseline] done index=13
[baseline] done index=14
[baseline] done index=15
[baseline] done index=16
[baseline] done index=17
[baseline] done index=18
[baseline] done index=19
[baseline] done index=20
[baseline] done index=21
[baseline] done index=22
[baseline] done index=23
[baseline] done index=24
[baseline] done index=25
[baseline] done index=26
[baseline] done index=27
[baseline] done index=28
[baseline] done index=29
[baseline] done index=30
[baseline] done index=31
[baseline] done index=32
[baseline] done index=33
[baseline] done index=34
[baseline] done index=35
[baseline] done index=36
[baseline] done index=37
[base

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.55s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [36]:
######################################################
# Chạy LLM-judge
######################################################

In [37]:
%%bash
if [ "$RUN_LLM_JUDGE" != "1" ]; then
  echo "RUN_LLM_JUDGE != 1, skip judge."
  exit 0
fi

export HF_HOME=/kaggle/tmp/megarag_vdoc/hf
export TRANSFORMERS_CACHE=/kaggle/tmp/megarag_vdoc/hf/models
export CUDA_VISIBLE_DEVICES=0
export PYTHONPATH=/kaggle/working/megarag_vdoc/MegaRAG

cd /kaggle/working/megarag_vdoc/MegaRAG

python egs/utils/judge_qwen.py \
  --eval-items /kaggle/working/megarag_outputs/eval_items.jsonl \
  --results /kaggle/working/megarag_outputs/results/results.jsonl \
  --output /kaggle/working/megarag_outputs/judges.jsonl \
  --sample ${JUDGE_SAMPLE:-20}

echo "Done LLM judge."

[qwen_llm] Detected 1 GPU(s): ['Tesla T4']
Saved judge results to /kaggle/working/megarag_outputs/judges.jsonl
Done LLM judge.


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.44s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [38]:
######################################################
# Cal metrics -> EM, Token F1, Recall + Table
######################################################

In [40]:
import os
import json
import re
import unicodedata
import pathlib
from collections import Counter
import pandas as pd

OUT = pathlib.Path(os.environ["OUT_DIR"])
results_path = OUT / "results" / "results.jsonl"
baseline_path = OUT / "results" / "baseline_results.jsonl"
eval_items_path = OUT / "eval_items.jsonl"
judges_path = OUT / "judges.jsonl"
metrics_path = OUT / "metrics.json"
predictions_path = OUT / "predictions.csv"


def normalize_answer(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"```.*?```", " ", s, flags=re.S)
    s = re.sub(r"[*_#`>|]", " ", s)
    s = unicodedata.normalize("NFC", s.lower())
    s = re.sub(r"\s+", " ", s).strip()
    return s


def tokenize(s):
    return re.findall(r"\w+", s, flags=re.UNICODE)


def token_f1(pred, gold):
    p_toks = tokenize(pred)
    g_toks = tokenize(gold)
    if not p_toks or not g_toks:
        return 0.0
    common = Counter(p_toks) & Counter(g_toks)
    num = sum(common.values())
    if num == 0:
        return 0.0
    p = num / len(p_toks)
    r = num / len(g_toks)
    return 2 * p * r / (p + r)


def token_recall(pred, gold):
    p_toks = tokenize(pred)
    g_toks = tokenize(gold)
    if not g_toks:
        return 0.0
    common = Counter(p_toks) & Counter(g_toks)
    num = sum(common.values())
    return num / len(g_toks)


def count_cuda_errors(text):
    if not text:
        return 0
    return str(text).lower().count("cuda out of memory") + str(text).lower().count("outofmemoryerror")


def load_jsonl(path):
    out = {}
    if not pathlib.Path(path).exists():
        return out
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            out[rec.get("index")] = rec
    return out


eval_items = []
with open(eval_items_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            eval_items.append(json.loads(line))

pred_map = load_jsonl(results_path)
baseline_map = load_jsonl(baseline_path)

judge_map = {}
if judges_path.exists():
    with open(judges_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                judge_map[rec["index"]] = rec.get("verdict", "NO")

rows = []

for item in eval_items:
    idx = item["index"]
    question = item["question"]
    gold = item["gold_answer"]

    # MegaRAG prediction
    megareg_rec = pred_map.get(idx, {})
    megareg_pred = "" if megareg_rec.get("error") else megareg_rec.get("answer", "")

    # Baseline prediction
    base_rec = baseline_map.get(idx, {})
    base_pred = "" if base_rec.get("error") else base_rec.get("answer", "")

    norm_gold = normalize_answer(gold)

    row = {
        "index": idx,
        "page_id": item.get("page_id", ""),
        "question": question,
        "gold_answer": gold,
        "megarag_answer": megareg_pred,
        "baseline_answer": base_pred,
        "megarag_em": int(norm_gold != "" and normalize_answer(megareg_pred) == norm_gold),
        "megarag_f1": token_f1(normalize_answer(megareg_pred), norm_gold),
        "megarag_recall": token_recall(normalize_answer(megareg_pred), norm_gold),
        "baseline_em": int(norm_gold != "" and normalize_answer(base_pred) == norm_gold),
        "baseline_f1": token_f1(normalize_answer(base_pred), norm_gold),
        "baseline_recall": token_recall(normalize_answer(base_pred), norm_gold),
        "megarag_cuda_errors": count_cuda_errors(megareg_pred),
    }

    if idx in judge_map:
        row["llm_judge"] = judge_map[idx]

    rows.append(row)

df = pd.DataFrame(rows)

metrics = {
    "num_eval": len(df),
    # MegaRAG
    "megarag_exact_match": float(df["megarag_em"].mean()) if len(df) else 0.0,
    "megarag_token_f1": float(df["megarag_f1"].mean()) if len(df) else 0.0,
    "megarag_token_recall": float(df["megarag_recall"].mean()) if len(df) else 0.0,
    "megarag_cuda_error_answers": int((df["megarag_cuda_errors"] > 0).sum()),
    # Baseline
    "baseline_exact_match": float(df["baseline_em"].mean()) if len(df) else 0.0,
    "baseline_token_f1": float(df["baseline_f1"].mean()) if len(df) else 0.0,
    "baseline_token_recall": float(df["baseline_recall"].mean()) if len(df) else 0.0,
}

if "llm_judge" in df.columns:
    judged = df[df["llm_judge"].isin(["YES", "NO"])]
    if len(judged) > 0:
        metrics["megarag_llm_judge_accuracy"] = float((judged["llm_judge"] == "YES").mean())
        metrics["num_llm_judged"] = int(len(judged))

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

df.to_csv(predictions_path, index=False)

print(json.dumps(metrics, ensure_ascii=False, indent=2))

# Bảng so sánh cuối
comparison = pd.DataFrame({
    "Method": ["MegaRAG + Qwen3-VL-8B", "Baseline Qwen3-VL-8B (no RAG)"],
    "EM": [metrics["megarag_exact_match"], metrics["baseline_exact_match"]],
    "Token F1": [metrics["megarag_token_f1"], metrics["baseline_token_f1"]],
    "Token Recall": [metrics["megarag_token_recall"], metrics["baseline_token_recall"]],
})

if "megarag_llm_judge_accuracy" in metrics:
    comparison["LLM-Judge Acc"] = [metrics.get("megarag_llm_judge_accuracy", float("nan")), float("nan")]

print("\nBẢNG SO SÁNH CUỐI")
print(comparison.to_string(index=False))

{
  "num_eval": 90,
  "megarag_exact_match": 0.0,
  "megarag_token_f1": 0.407995656903226,
  "megarag_token_recall": 0.66992362727871,
  "megarag_cuda_error_answers": 0,
  "baseline_exact_match": 0.0,
  "baseline_token_f1": 0.23630925583348653,
  "baseline_token_recall": 0.1484197024562618,
  "megarag_llm_judge_accuracy": 0.4888888888888889,
  "num_llm_judged": 90
}

BẢNG SO SÁNH CUỐI
                       Method  EM  Token F1  Token Recall  LLM-Judge Acc
        MegaRAG + Qwen3-VL-8B 0.0  0.407996      0.669924       0.488889
Baseline Qwen3-VL-8B (no RAG) 0.0  0.236309      0.148420            NaN
